In [ ]:
"""
GEN5 TRAINING — A + B
=====================
Option A: rigorous scaling baseline on noisy sine (same as scaling_compare.py
          but with Gen5 added). Likely shows Gen5 doesn't earn its keep on
          single-axis structure. Establishes the negative honestly if so.

Option B: structured-task test. 3-channel signal (3 independent waveforms),
          target is next-step of all 3. This is the task composition was
          designed for: Gen3's p-extensions can specialize one per channel,
          Gen5 adds saddle + edge on top. If Gen5 still loses on this, the
          architecture genuinely doesn't deliver. If it wins, real evidence.

Run:
    python3 train_gen5.py

Output: two tables (sine + 3-channel) + scaling exponents + JSON.
Wall: ~10-15 minutes on CPU. Reduce SEEDS or TARGETS at bottom if too slow.
"""

import math, json, time, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(2)


# =====================================================================
# TASK A — noisy sine (single channel)
# =====================================================================
def make_sine(B, T, device='cpu', noise=0.3):
    t = torch.linspace(0, 6*math.pi, T+1, device=device).unsqueeze(0).expand(B, -1)
    ph = torch.rand(B, 1, device=device) * 2*math.pi
    clean = torch.sin(t + ph)
    n = torch.randn_like(clean) * noise
    return (clean+n)[:, :-1].unsqueeze(-1), clean[:, 1:].unsqueeze(-1)


# =====================================================================
# TASK B — 3-channel structured signal
# =====================================================================
# Three independent channels:
#   ch0: low-freq sine     (slow, clean)
#   ch1: mid-freq sine     (moderate freq, moderate noise)
#   ch2: high-freq sawtooth (fast, jagged, distinct shape)
# Target: predict next-step of all 3 channels.
# This task has explicit multi-axis structure that Gen3's p-extensions
# can specialize on, that single-substrate models have to share over.

def make_multichannel(B, T, device='cpu'):
    t = torch.linspace(0, 6*math.pi, T+1, device=device).unsqueeze(0).expand(B, -1)
    ph0 = torch.rand(B, 1, device=device) * 2*math.pi
    ph1 = torch.rand(B, 1, device=device) * 2*math.pi
    ph2 = torch.rand(B, 1, device=device) * 2*math.pi

    ch0 = torch.sin(0.7 * t + ph0)
    ch1 = torch.sin(2.0 * t + ph1)
    # sawtooth via mod
    saw_t = (3.5 * t + ph2) / (2*math.pi)
    ch2 = 2 * (saw_t - torch.floor(saw_t + 0.5))

    clean = torch.stack([ch0, ch1, ch2], dim=-1)   # (B, T+1, 3)
    n = torch.randn_like(clean) * 0.2
    x = (clean + n)[:, :-1, :]
    y = clean[:, 1:, :]
    return x, y


# =====================================================================
# LAYER PRIMITIVES
# =====================================================================
def sphere_proj(h, r):
    nrm = h.norm(dim=-1, keepdim=True).clamp_min(1e-6)
    return torch.tanh(nrm / r) * r * (h / nrm)

def cube_proj(h, r):
    return torch.tanh(h / r) * r

def smooth_max(h, beta=8.0):
    sat = h.abs().clamp(0, 5)
    return (1.0 / beta) * torch.logsumexp(beta * sat, dim=-1, keepdim=True)


# =====================================================================
# ARCHITECTURES (in_dim parameterized for multi-channel)
# =====================================================================

class Gen0_Vanilla(nn.Module):
    def __init__(self, H, in_dim=1, out_dim=1):
        super().__init__()
        self.rnn = nn.RNN(in_dim, H, 1, nonlinearity='tanh', batch_first=True)
        self.head = nn.Linear(H, out_dim)
    def forward(self, x): return self.head(self.rnn(x)[0])
    @staticmethod
    def rec_params(H): return H*H

class Gen1_Single(nn.Module):
    def __init__(self, H, in_dim=1, out_dim=1, r=1.0):
        super().__init__()
        self.H, self.r = H, r
        self.inp = nn.Linear(in_dim, H)
        self.W = nn.Linear(H, H, bias=False)
        self.head = nn.Linear(H, out_dim)
    def _bound(self, h): return sphere_proj(h, self.r)
    def forward(self, x):
        B, T, _ = x.shape
        h = torch.zeros(B, self.H, device=x.device)
        outs = []
        for ti in range(T):
            h = self._bound(self.W(h) + self.inp(x[:, ti]))
            outs.append(self.head(h))
        return torch.stack(outs, 1)
    @staticmethod
    def rec_params(H): return H*H

class Gen3_Composed(nn.Module):
    def __init__(self, H, in_dim=1, out_dim=1, r=1.0, couple=0.3):
        super().__init__()
        self.H, self.r, self.couple = H, r, couple
        self.inp = nn.Linear(in_dim, H)
        self.W_s  = nn.Linear(H, H, bias=False)
        self.W_px = nn.Linear(H, H, bias=False)
        self.W_py = nn.Linear(H, H, bias=False)
        self.W_pz = nn.Linear(H, H, bias=False)
        self.head = nn.Linear(H, out_dim)
    def _bound(self, h): return sphere_proj(h, self.r)
    def forward(self, x):
        B, T, _ = x.shape
        s  = torch.zeros(B, self.H, device=x.device)
        px = torch.zeros_like(s); py = torch.zeros_like(s); pz = torch.zeros_like(s)
        outs = []
        for ti in range(T):
            u = self.inp(x[:, ti])
            px = self._bound(self.W_px(px) + u - self.couple*(px - s))
            py = self._bound(self.W_py(py) + u - self.couple*(py - s))
            pz = self._bound(self.W_pz(pz) + u - self.couple*(pz - s))
            s  = self._bound(self.W_s(s) + (px+py+pz)/3)
            outs.append(self.head(s))
        return torch.stack(outs, 1)
    @staticmethod
    def rec_params(H): return 4*H*H

class Gen5_FullScope(nn.Module):
    """Eight-layer full-scope substrate. Same code path as gen5_full_scope.py
    but parameterized for variable in_dim / out_dim to support 3-channel task."""
    def __init__(self, H, in_dim=1, out_dim=1, r_outer=1.0, r_inner=0.65, R_dodec=1.2):
        super().__init__()
        self.H = H
        self.r_outer, self.r_inner, self.R_dodec = r_outer, r_inner, R_dodec
        self.inp = nn.Linear(in_dim, H)
        self.W_s  = nn.Linear(H, H, bias=False)
        self.W_px = nn.Linear(H, H, bias=False)
        self.W_py = nn.Linear(H, H, bias=False)
        self.W_pz = nn.Linear(H, H, bias=False)
        self.couple   = nn.Parameter(torch.tensor(0.30))
        self.pi_dyn   = nn.Parameter(torch.tensor(4.0))
        self.phi_dyn  = nn.Parameter(torch.tensor(0.70))
        self.vesica_a = nn.Parameter(torch.tensor(0.40))
        self.vesica_b = nn.Parameter(torch.tensor(0.40))
        self.head = nn.Linear(H, out_dim)
    def _paired(self, h):
        sat = smooth_max(h)
        a = torch.sigmoid(self.pi_dyn*(sat - self.phi_dyn))
        sp = sphere_proj(h, self.r_inner)
        cu = cube_proj(h, self.r_outer)
        return (1.0-a)*sp + a*cu
    def _vesica(self, m):
        return -(-self.vesica_a*m + self.vesica_b*m**3)
    def _dodec(self, h):
        return torch.clamp(h, -self.R_dodec, self.R_dodec)
    def forward(self, x):
        B, T, _ = x.shape
        s  = torch.zeros(B, self.H, device=x.device)
        px = torch.zeros_like(s); py = torch.zeros_like(s); pz = torch.zeros_like(s)
        outs = []
        for ti in range(T):
            u = self.inp(x[:, ti])
            px_raw = self.W_px(px) + u - self.couple*(px - s) + self._vesica(px)*0.1
            py_raw = self.W_py(py) + u - self.couple*(py - s) + self._vesica(py)*0.1
            pz_raw = self.W_pz(pz) + u - self.couple*(pz - s) + self._vesica(pz)*0.1
            px = self._dodec(self._paired(px_raw))
            py = self._dodec(self._paired(py_raw))
            pz = self._dodec(self._paired(pz_raw))
            s  = self._dodec(self._paired(self.W_s(s) + (px+py+pz)/3))
            outs.append(self.head(s))
        return torch.stack(outs, 1)
    @staticmethod
    def rec_params(H): return 4*H*H


# =====================================================================
# HARNESS
# =====================================================================
def train_once(model, task_fn, steps=300, B=16, T=48, lr=3e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    t0 = time.time()
    for step in range(steps):
        x, y = task_fn(B, T)
        loss = F.mse_loss(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(float(loss.item()))
    wall = time.time() - t0
    final = float(np.mean(losses[-20:]))
    best = float(min(losses[20:]) if len(losses) > 20 else min(losses))
    return final, best, wall

def pick_H(arch_cls, target):
    H = 1
    while arch_cls.rec_params(H) < target: H += 1
    return H


# =====================================================================
# CONFIG
# =====================================================================
ARCHS = {
    'Gen0_Vanilla':   Gen0_Vanilla,
    'Gen1_Single':    Gen1_Single,
    'Gen3_Composed':  Gen3_Composed,
    'Gen5_FullScope': Gen5_FullScope,
}
TARGETS = [500, 2000, 8000]
SEEDS = [0, 1, 2]


def run_task(task_name, task_fn, in_dim, out_dim):
    print("=" * 78)
    print(f"TASK: {task_name}    in_dim={in_dim}  out_dim={out_dim}")
    print("=" * 78)
    results = {}
    overall_t0 = time.time()
    for arch_name, arch_cls in ARCHS.items():
        results[arch_name] = []
        for target in TARGETS:
            H = pick_H(arch_cls, target)
            actual = arch_cls.rec_params(H)
            finals, bests, walls = [], [], []
            for seed in SEEDS:
                torch.manual_seed(seed); np.random.seed(seed)
                model = arch_cls(H=H, in_dim=in_dim, out_dim=out_dim)
                f, b, w = train_once(model, task_fn)
                finals.append(f); bests.append(b); walls.append(w)
            row = {
                'H': H, 'rec_params': actual, 'target': target,
                'final_loss_mean': float(np.mean(finals)),
                'final_loss_std':  float(np.std(finals)),
                'best_loss_mean':  float(np.mean(bests)),
                'best_loss_std':   float(np.std(bests)),
                'wall_mean':       float(np.mean(walls)),
            }
            results[arch_name].append(row)
            print(f"  {arch_name:<18} H={H:<4} params={actual:<6} "
                  f"best={row['best_loss_mean']:.4f}±{row['best_loss_std']:.4f}  "
                  f"final={row['final_loss_mean']:.4f}  "
                  f"wall={row['wall_mean']:.1f}s")
        print()
    print(f"task wall: {time.time()-overall_t0:.1f}s\n")

    # summary table
    print(f"SUMMARY · {task_name}  (best-loss mean)")
    print(f"{'arch':<20} | " + " | ".join([f"~{t}p".rjust(16) for t in TARGETS]))
    print("-" * 78)
    for name, rows in results.items():
        cells = [f"{r['best_loss_mean']:.4f}±{r['best_loss_std']:.4f}".rjust(16) for r in rows]
        print(f"{name:<20} | " + " | ".join(cells))
    print()

    # scaling exponents
    print(f"SCALING EXPONENT alpha · {task_name}")
    for name, rows in results.items():
        xs = np.log([r['rec_params'] for r in rows])
        ys = np.log([r['best_loss_mean'] for r in rows])
        slope = float(np.polyfit(xs, ys, 1)[0])
        print(f"  {name:<20}  alpha = {-slope:+.3f}   "
              f"({rows[0]['best_loss_mean']:.4f} -> {rows[-1]['best_loss_mean']:.4f})")
    print()
    return results


# =====================================================================
# RUN BOTH OPTIONS
# =====================================================================
all_results = {}

# OPTION A
print("\n" + "★" * 78)
print("OPTION A — RIGOROUS BASELINE · NOISY SINE (single-axis)")
print("★" * 78)
all_results['A_sine'] = run_task('NOISY_SINE', make_sine, in_dim=1, out_dim=1)

# OPTION B
print("\n" + "★" * 78)
print("OPTION B — STRUCTURED TASK · 3-CHANNEL (multi-axis)")
print("★" * 78)
all_results['B_3channel'] = run_task('3-CHANNEL', make_multichannel, in_dim=3, out_dim=3)

# head-to-head summary
print("=" * 78)
print("HEAD-TO-HEAD: does Gen5 beat Gen1 / Gen3 on each task?")
print("=" * 78)
for task_key, results in all_results.items():
    print(f"\n{task_key}:")
    for i, target in enumerate(TARGETS):
        g1 = results['Gen1_Single'][i]['best_loss_mean']
        g3 = results['Gen3_Composed'][i]['best_loss_mean']
        g5 = results['Gen5_FullScope'][i]['best_loss_mean']
        winner = min([('Gen1', g1), ('Gen3', g3), ('Gen5', g5)], key=lambda x: x[1])
        print(f"  ~{target}p:  Gen1={g1:.4f}  Gen3={g3:.4f}  Gen5={g5:.4f}  "
              f"-> winner: {winner[0]} ({winner[1]:.4f})")

with open('gen5_training_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print("\nSaved gen5_training_results.json")
print()
print("READING THE RESULT:")
print("  - On TASK A (sine): expected Gen1 wins. If Gen5 surprises by winning,")
print("    composition has value even on single-axis structure.")
print("  - On TASK B (3-channel): this is composition's test. If Gen3 or Gen5")
print("    wins, the architecture earns its keep on multi-axis structure.")
print("    If Gen1 still wins, the eight-layer story has no benchmark backing.")


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
OPTION A — RIGOROUS BASELINE · NOISY SINE (single-axis)
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
TASK: NOISY_SINE    in_dim=1  out_dim=1
  Gen0_Vanilla       H=23   params=529    best=0.0237±0.0001  final=0.0293  wall=4.6s
  Gen0_Vanilla       H=45   params=2025   best=0.0225±0.0008  final=0.0284  wall=2.1s
  Gen0_Vanilla       H=90   params=8100   best=0.0211±0.0014  final=0.0287  wall=3.3s

  Gen1_Single        H=23   params=529    best=0.0210±0.0006  final=0.0272  wall=7.2s
  Gen1_Single        H=45   params=2025   best=0.0201±0.0001  final=0.0264  wall=7.1s
  Gen1_Single        H=90   params=8100   best=0.0189±0.0005  final=0.0267  wall=10.1s

  Gen3_Composed      H=12   params=576    best=0.0220±0.0002  final=0.0274  wall=24.0s
  Gen3_Composed      H=23   params=2116   best=0.0209±0.0006  final=0.0265  wall=25.2s
  Gen3_Composed      H=45   params=8100   best=0.020